# Statistics for Data Analysts/Engineers — Module 14: A/B Testing in Practice

In Modules 6, 7, and 9 we already ANALYZED ready-made A/B test data (`campaign_group`
and `converted`). This module covers the other side of the coin: how to DESIGN such a
test well from the very start (how many observations you need, how much power it has to
detect an effect) and how to avoid the most common practical pitfalls that ruin
conclusions even from a correctly computed test.

## Table of contents
1. [Anatomy of a good A/B test](#sec1)
2. [Sample size and statistical power](#sec2)
3. [How many observations do you need?](#sec3)
4. [How much power did our test have?](#sec4)
5. [Statistical vs. practical significance](#sec5)
6. [Pitfall: peeking at results](#sec6)
7. [Pitfall: multiple comparisons](#sec7)
8. [Summary and exercises](#sec8)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. Anatomy of a good A/B test

A well-designed A/B test has several properties, all decided BEFORE any data is
collected:

1. **Random assignment to groups** -- exactly what gave us causation (not just
   correlation) in Module 11: both groups should have a similar distribution of every
   trait except the one factor being tested.
2. **One, pre-decided primary metric** -- for us, `converted`. Checking many metrics at
   once and picking whichever one "came out" significant is a version of the multiple
   comparisons problem from Module 10 (we return to this in section 7).
3. **A pre-decided sample size** (or a formal stopping rule) -- based on a statistical
   power analysis (sections 2-4), NOT a decision made "by eye" while data is being
   collected (section 6).
4. **A pre-decided significance level** `alpha` (usually `0.05`) and desired **power**
   (usually `0.8`).

<a id="sec2"></a>
## 2. Sample size and statistical power

**Statistical power** is the probability of detecting an effect, IF it truly exists --
that is, `1 - beta`, where `beta` is the Type II error risk from Module 7 (failing to
reject H0 even though H1 is true). Power depends on three things at once:

- **effect size** -- the bigger the true difference between groups, the easier it is to
  detect,
- **sample size** -- more observations means less noise, making it easier to detect even
  a small effect,
- **the `alpha` level** -- the stricter the significance threshold, the harder it is to
  detect anything with the same sample.

For comparing two proportions (exactly our case: `converted` in group A vs. B),
`statsmodels` provides ready-made tools for this analysis.

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

power_analysis = NormalIndPower()


<a id="sec3"></a>
## 3. How many observations do you need?

Before collecting any data, we can ask: "how many observations per group do I need to
detect a conversion difference from `10%` to `20%`, with power `0.8` at `alpha=0.05`?".

In [ ]:
effect_size = proportion_effectsize(0.20, 0.10)
print(f"Effect size (Cohen's h for proportions): {effect_size:.4f}")

n_needed = power_analysis.solve_power(
    effect_size=effect_size, alpha=0.05, power=0.8, ratio=1.0, alternative="two-sided"
)
print(f"Observations needed per group: {n_needed:.0f}")


> 📊 **proportion_effectsize computes Cohen's h**
>
> `proportion_effectsize` converts two proportions into a single number describing the "distance" between them on a special, normalized (arcsine) scale, independent of sample size -- exactly the way Cohen's d from Module 8 did it for a difference of means.

> ⚠️ **A small effect needs a MUCH larger sample**
>
> Detecting a subtle difference requires far more data than detecting a large one -- you'll see this yourself in exercise 1 below for a `10%` -> `15%` difference instead of `10%` -> `20%`. That's why minor UX tweaks at large companies get tested on millions of users, not a few hundred.

<a id="sec4"></a>
## 4. How much power did our test have?

We can also ask the reverse question: given a test we've already run (our `customers`
data), how much power did it actually have to detect the observed difference?

In [ ]:
conversion_rate = customers.groupby("campaign_group")["converted"].agg(["mean", "count"])
print(conversion_rate)

observed_effect_size = proportion_effectsize(
    conversion_rate.loc["B", "mean"], conversion_rate.loc["A", "mean"]
)
achieved_power = power_analysis.solve_power(
    effect_size=observed_effect_size,
    nobs1=conversion_rate.loc["A", "count"],
    alpha=0.05,
    ratio=conversion_rate.loc["B", "count"] / conversion_rate.loc["A", "count"],
    alternative="two-sided",
)
print(f"Achieved power: {achieved_power:.3f}")


Power above `0.8` (the conventional "sufficient power" threshold) means our test had a
solid chance of detecting an effect of this size, if it truly existed -- which fits with
it actually coming out significant in Module 9.

<a id="sec5"></a>
## 5. Statistical vs. practical significance

A reminder from Modules 7 and 11: **statistical significance** (a p-value below `alpha`)
is not the same as **practical significance** (whether the effect is large enough to
matter for the business). With a very large sample, even a conversion difference from
`10.0%` to `10.3%` can come out "statistically significant" -- but whether it's worth
rolling out a change for `0.3` percentage points is a BUSINESS question, not a
statistical one. Always report together: the p-value, the effect size (here: the
percentage-point difference or the odds ratio from Module 13), and the confidence
interval (Module 6) for that difference.

<a id="sec6"></a>
## 6. Pitfall: peeking at results

**Peeking** (*optional stopping*) means checking the p-value MANY times while data is
still being collected, and stopping the test as soon as it comes out significant. Let's
simulate (exactly like in Module 7) what this does to the real Type I error risk when H0
is TRUE (the groups are genuinely identical):

In [ ]:
from scipy import stats

rng2 = np.random.default_rng(7)
num_simulations = 2000
n_max = 400
checkpoints = [50, 100, 150, 200, 250, 300, 350, 400]

false_alarms_once = 0
false_alarms_peeking = 0

for _ in range(num_simulations):
    # H0 is true: both groups come from the same distribution
    data_a = rng2.normal(0, 1, size=n_max)
    data_b = rng2.normal(0, 1, size=n_max)

    # (a) a single test at the very end
    _, p_final = stats.ttest_ind(data_a, data_b)
    if p_final < 0.05:
        false_alarms_once += 1

    # (b) a test at every checkpoint, stopping at the first significant result
    for k in checkpoints:
        _, p_k = stats.ttest_ind(data_a[:k], data_b[:k])
        if p_k < 0.05:
            false_alarms_peeking += 1
            break

print(f"Single test at the end: {false_alarms_once / num_simulations:.3f} false alarms")
print(f"Repeated peeking ({len(checkpoints)} checkpoints): {false_alarms_peeking / num_simulations:.3f} false alarms")


> ⚠️ **Peeking can more than triple the real Type I error risk**
>
> Notice: with H0 true, a single test at the end gives a real risk close to the stated `5%` (`alpha=0.05`) -- exactly matching the definition from Module 7. But checking the result at each of eight checkpoints and stopping at the first significant one raises the real risk to roughly `18%` -- this is the same multiple comparisons problem from Module 10, just spread across time instead of across variables. Solutions: set the sample size in advance (section 3) and stick to it, or use a formal, corrected sequential method (e.g. a Bonferroni correction -- see the challenge below).

<a id="sec7"></a>
## 7. Pitfall: multiple comparisons

This is exactly the same problem from Module 10, section 1, applied to A/B tests: if we
check MANY metrics at once (conversion, satisfaction, time on page, cart value, ...) and
call the test a "success" whenever ANY ONE of them comes out significant, the real risk
of a false alarm grows exactly the same way it did with many simultaneous t-tests.

In [ ]:
num_metrics = 5
false_alarm_risk = 1 - (1 - 0.05) ** num_metrics
print(f"Risk of at least one false alarm with {num_metrics} metrics: {false_alarm_risk:.3f}")


<a id="sec8"></a>
## 8. Summary and exercises

In this module we covered:
- the key elements of a well-designed A/B test (random assignment, one primary metric,
  a pre-decided sample size),
- statistical power and its dependence on effect size, sample size, and `alpha`,
- computing required sample size and achieved power with `statsmodels.stats.power`,
- the difference between statistical and practical significance,
- the peeking (optional stopping) pitfall and its effect on the real Type I error risk,
- the multiple comparisons problem applied to several metrics in an A/B test.

> 📝 **Exercise 1: Sample size for a smaller effect**
>
> Compute the sample size per group needed to detect a conversion difference from `10%` to `15%` (instead of `10%` to `20%` from section 3), at the same power `0.8` and `alpha=0.05`. Compare with the result from section 3.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
small_effect_size = proportion_effectsize(0.15, 0.10)
n_needed_small = power_analysis.solve_power(
    effect_size=small_effect_size, alpha=0.05, power=0.8, ratio=1.0, alternative="two-sided"
)
print(f"Observations needed per group (10% -> 15%): {n_needed_small:.0f}")
```

This comes out to about 680 observations per group -- more than 3.5 times the roughly 195 needed to detect the larger 10% -> 20% difference from section 3. A smaller effect means a substantially larger required sample, exactly as in the warn() box above.
</details>

> 📝 **Exercise 2: Power with a smaller sample**
>
> Compute the achieved power for the effect from section 3 (`10%` vs `20%`), but with a much smaller sample: `nobs1=50` per group. Compare it with the power `0.8` we originally planned for.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
power_at_50 = power_analysis.solve_power(
    effect_size=effect_size, nobs1=50, alpha=0.05, ratio=1.0, alternative="two-sided"
)
print(f"Power with 50 observations per group: {power_at_50:.3f}")
```
</details>

> 📝 **Exercise 3: Fewer checkpoints, smaller inflation**
>
> Repeat the peeking simulation from section 6, but with only TWO checkpoints (e.g. `[200, 400]`) instead of eight. Is the inflation of the real Type I error risk smaller than with eight checkpoints?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
rng3 = np.random.default_rng(7)
false_alarms_2pt = 0
for _ in range(num_simulations):
    data_a = rng3.normal(0, 1, size=n_max)
    data_b = rng3.normal(0, 1, size=n_max)
    for k in [200, 400]:
        _, p_k = stats.ttest_ind(data_a[:k], data_b[:k])
        if p_k < 0.05:
            false_alarms_2pt += 1
            break
print(f"2 checkpoints: {false_alarms_2pt / num_simulations:.3f} false alarms")
```

The real risk with 2 checkpoints comes out lower than with 8 (but still higher than the nominal 5%) -- the more times you 'peek', the bigger the inflation of the false-alarm risk.
</details>

> 📝 **Exercise 4: False-alarm risk with many metrics**
>
> Compute the risk of at least one false alarm (section 7) for `10` simultaneously checked metrics instead of `5`. How much does the risk increase?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
num_metrics_10 = 10
risk_10 = 1 - (1 - 0.05) ** num_metrics_10
print(f"Risk with 10 metrics: {risk_10:.3f}")
```
</details>

> 🔥 **Challenge: Bonferroni correction for peeking**
>
> Repeat the peeking simulation from section 6 (8 checkpoints), but this time compare the p-value against a CORRECTED threshold `alpha_bonferroni = 0.05 / num_checkpoints` instead of plain `0.05` (the exact same correction logic used for multiple comparisons in Module 10). Does the real Type I error risk come back down near the nominal `5%`?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
rng4 = np.random.default_rng(7)
alpha_bonferroni = 0.05 / len(checkpoints)
false_alarms_bonferroni = 0

for _ in range(num_simulations):
    data_a = rng4.normal(0, 1, size=n_max)
    data_b = rng4.normal(0, 1, size=n_max)
    for k in checkpoints:
        _, p_k = stats.ttest_ind(data_a[:k], data_b[:k])
        if p_k < alpha_bonferroni:
            false_alarms_bonferroni += 1
            break

print(f"Alpha after Bonferroni correction: {alpha_bonferroni:.5f}")
print(f"False alarms with correction: {false_alarms_bonferroni / num_simulations:.3f}")
```

The corrected threshold (about 0.00625 instead of 0.05) brings the real false-alarm risk back down near the nominal 5% (even slightly below, since the Bonferroni correction is inherently a bit conservative). This works, but at the cost of needing a larger effect to detect anything at any single checkpoint -- in practice, dedicated sequential-testing methods are used more often (beyond the scope of this course).
</details>

## What's next?

This is the second-to-last module of the course! In **Module 15**, the FINAL module,
we'll bring everything together: descriptive statistics, hypothesis testing, regression,
and A/B testing best practices, all in one complete, end-to-end analysis of a business
question -- from the first look at the data, through formulating a hypothesis, to a
final report with conclusions.